# Copert : moyenne et variance et méthodes de calcul

Dans ce notebook est analysé l'impact de deux méthodes de calcul de vitesse sur l'estimation des émissions via Copert.

## 1 - Environnement

### 1.1 - Dépendances

In [ ]:
# pip

%pip install --upgrade pip

In [ ]:
# installation des dépendances

%pip install --upgrade sys
%pip install --upgrade os

%pip install --upgrade pathlib

%pip install --upgrade importlib

%pip install --upgrade numpy
%pip install --upgrade pandas

%pip install --upgrade shapely

%pip install --upgrade ast

%pip install --upgrade logging

In [ ]:
# chargement des dépendances

import sys
import os

from pathlib import Path

import importlib

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely import wkt
from shapely.geometry import Point
from shapely.geometry import Polygon

import ast

import logging

In [ ]:
# configuration du logging

logging.basicConfig(
    level=logging.INFO,  
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

In [ ]:
# chemin relatif depuis ce notebook jusqu'au dossier parent de modules 

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

### 1.2 - Classes et fonctions

In [ ]:
# import des classes

import modules.CopertEstimator
import modules.DemandVariationCarSimulation1Res
import modules.Utils as Utils

importlib.reload(sys.modules['modules.CopertEstimator'])
importlib.reload(sys.modules['modules.DemandVariationCarSimulation1Res'])
importlib.reload(Utils)

CopertEstimator = modules.CopertEstimator.CopertEstimator
DemandVariationCarSimulation1Res = modules.DemandVariationCarSimulation1Res.DemandVariationCarSimulation1Res

## 2 - Contexte, chargement, prétraitement

### 2.1 - Contexte

La classe *CopertEstimator* a pour rôle de calculer les émissions de NOX et de CO2 produites par un trafic simulé dans MnMS. En pratique, une instance de *CopertEstimator* est construite en association avec des fichiers de sorties de simulations à partir desquels sont estimés les émissions d'abords par véhicule, puis par l'agrégation spatio-temporelle et catégorielle souhaitée. *copert_estimation(period:str, method:str)* est la fonction responsable de la première étape de ce processus. Pour être plus précis, la fonction effectue une agrégation temporelle par véhicule puis calcule la distance parcourue et la vitesse moyenne observée pendant cette période. Ces informations sont ensuite utilisés pour estimer les émissions avec Copert. Elle prend en paramètres les arguments *period* et *method* dont les valeurs contrôlent la méthode d'estimation des émissions. *period* est la période d'agrégation temporelle, *method*  est lé méthode d'estimation de la vitesse, *SPEED_MEAN* calcule la vitesse par moyenne des vitesses imposées par le MFD pendant la période d'aggrégation, *DISTANCE_DIFF* calcule la vitesse comme rapport entre la distance parcourue pendant la période d'aggrégation et cette dernière.
Il paraît intéressant de chercher l'impact de ces deux arguments sur le calcul des émissions, de comprendre l'origine de ces différences puis de déterminer les meilleurs combinaisons de paramètres.


### 2.2 - Chargement et prétraitement

In [ ]:
# chargement des demandes

chemin_demande_matsim =  "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__1.00__1_1.00.csv"
demande_matsim = pd.read_csv(chemin_demande_matsim, sep=';')

chemin_demande_amsterdam = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00.csv"
demande_amsterdam = pd.read_csv(chemin_demande_amsterdam, sep=';') 

# 3 - Simulations

In [ ]:
# fichiers reseaux

graph_path = "../../network/graph/Amsterdam__Matsim__graph.json"
od_layer_path = "../../network/test_files/Amsterdam__Matsim__graph_od_layer_14_16_1000.json"
transit_link_path = "../../network/test_files/Amsterdam__Matsim__graph_transit_link_14_16_1000_grid.json"
a = 2.8981940480441857
b = -0.00010553060526140915
MFD_variables = [a, b]

# demandes

chemin_demande_matsim = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original"
chemin_demande_amsterdam = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original"

In [ ]:
# simulations pour la demande Matsim_7-9 calibrée

output_directory = "../../outputs/Matsim_7-9/simulations/Matsim_7-9_Calibration__Original__Simulation_Dummy__14_16_1000"

DVCS1R = DemandVariationCarSimulation1Res(chemin_demande_matsim, graph_path)
DVCS1R.simulate(od_layer_path, transit_link_path, MFD_variables, "Dummy", output_directory)

In [ ]:
# simulations pour la demande AmsterdamCity_7-9 originale

output_directory = "../../outputs/AmsterdamCity_7-9/simulations/AmsterdamCity_7-9_Original__Original__Simulation_Dummy__14_16_1000"

DVCS1R = DemandVariationCarSimulation1Res(chemin_demande_amsterdam, graph_path)
DVCS1R.simulate(od_layer_path, transit_link_path, MFD_variables, "Dummy", output_directory)

# 4 - Copert : méthodes de calculs

Dans un premier temps, les estimations vont être effectuées pour chaque simulation dans leur totalité pour plusieurs période d'aggrégation temporelle et pour chaque méthode en discriminant par demande source. 

## 4.1 - Matsim_7-9

In [ ]:
# parameters analysis for Matsim_7-9

simulation_outputs_directory="../../outputs/Matsim_7-9/simulations/Matsim_7-9_Calibration__Original__Simulation_Dummy__14_16_1000/Matsim_7-9_Calibration__Original__None_None__None_None__None__1.00__1_1.00"
crs="epsg:32631"
copert_data_path="../../data/copert_data/test/FE_COPERT2025.csv"
matsim_CE = CopertEstimator(simulation_outputs_directory=simulation_outputs_directory, crs=crs, copert_data_path=copert_data_path)
matsim_analysis_df = matsim_CE.copert_parameters_analysis(periods=["3min","6min","10min","20min"])
matsim_analysis_df.to_csv("../../outputs/Matsim_7-9/analysis/Matsim_7-9_Calibration__Original__Copert_Parameters_analysis/copert_parameters_analysis.csv", sep=';', index=False)

In [ ]:
# affichage 

matsim_analysis_df = pd.read_csv("../../outputs/Matsim_7-9/analysis/Matsim_7-9_Calibration__Original__Copert_Parameters_analysis/copert_parameters_analysis.csv", sep=';')

distancediff_buffer = matsim_analysis_df.loc[matsim_analysis_df["METHOD"]=="DISTANCE_DIFF"]
speedmean_buffer = matsim_analysis_df.loc[matsim_analysis_df["METHOD"]=="SPEED_MEAN"]

colors = {"DISTANCE_DIFF":"red","SPEED_MEAN":"blue"}

fix, ax = plt.subplots(1,2,figsize=(10,10))

ax[0].plot(distancediff_buffer["PERIOD"], distancediff_buffer["NOX"], marker='o', label="DISTANCE_DIFF", color="red")
ax[0].plot(speedmean_buffer["PERIOD"], speedmean_buffer["NOX"], marker='o', label="SPEED_MEAN", color="blue")
ax[0].set_xlabel("PERIOD (h)")
ax[0].set_ylabel("NOX (g*km/h)")
ax[0].set_title("Parameters sensitivity for NOX copert estimations")
ax[0].legend()

ax[1].plot(distancediff_buffer["PERIOD"], distancediff_buffer["CO2"], marker='o', label="DISTANCE_DIFF", color="red")
ax[1].plot(speedmean_buffer["PERIOD"], speedmean_buffer["CO2"], marker='o', label="SPEED_MEAN", color="blue")
ax[1].set_xlabel("PERIOD (h)")
ax[1].set_ylabel("CO2 (g*km/h)")
ax[1].set_title("Parameters sensitivity for CO2 copert estimations")
ax[1].legend()


plt.show()

## 4.2 - Amsterdam_7-9

In [ ]:
# parameters analysis for AmsterdamCity_7-9

simulation_outputs_directory= "../../outputs/AmsterdamCity_7-9/simulations/AmsterdamCity_7-9_Original__Original__Simulation_Dummy__14_16_1000/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00"
crs="epsg:32631"
copert_data_path="../../data/copert_data/test/FE_COPERT2025.csv"
amsterdam_CE = CopertEstimator(simulation_outputs_directory=simulation_outputs_directory, crs=crs, copert_data_path=copert_data_path)
amsterdam_analysis_df = amsterdam_CE.copert_parameters_analysis(periods=["3min","6min","10min","20min"])
amsterdam_analysis_df.to_csv("../../outputs/AmsterdamCity_7-9/analysis/AmsterdamCity_7-9_Original__Original__Copert_Parameters_analysis/copert_parameters_analysis.csv", sep=';', index=False)

In [ ]:
# affichage 

amsterdam_analysis_df = pd.read_csv("../../outputs/AmsterdamCity_7-9/analysis/AmsterdamCity_7-9_Original__Original__Copert_Parameters_analysis/copert_parameters_analysis.csv", sep=';')

distancediff_buffer = matsim_analysis_df.loc[matsim_analysis_df["METHOD"]=="DISTANCE_DIFF"]
speedmean_buffer = matsim_analysis_df.loc[matsim_analysis_df["METHOD"]=="SPEED_MEAN"]

colors = {"DISTANCE_DIFF":"red","SPEED_MEAN":"blue"}

fix, ax = plt.subplots(1,2,figsize=(10,10))

ax[0].plot(distancediff_buffer["PERIOD"], distancediff_buffer["NOX"], marker='o', label="DISTANCE_DIFF", color="red")
ax[0].plot(speedmean_buffer["PERIOD"], speedmean_buffer["NOX"], marker='o', label="SPEED_MEAN", color="blue")
ax[0].set_xlabel("PERIOD (h)")
ax[0].set_ylabel("NOX (g*km/h)")
ax[0].set_title("Parameters sensitivity for NOX copert estimations")
ax[0].legend()

ax[1].plot(distancediff_buffer["PERIOD"], distancediff_buffer["CO2"], marker='o', label="DISTANCE_DIFF", color="red")
ax[1].plot(speedmean_buffer["PERIOD"], speedmean_buffer["CO2"], marker='o', label="SPEED_MEAN", color="blue")
ax[1].set_xlabel("PERIOD (h)")
ax[1].set_ylabel("CO2 (g*km/h)")
ax[1].set_title("Parameters sensitivity for CO2 copert estimations")
ax[1].legend()


plt.show()

## 5 - Petite analyse exploratoire

In [ ]:
# affichage vitesse SPEED_MEAN et DISTANCE_DIFF en fonction du temps, 3 min

Utils.display_car_speed_distance_time("../../outputs/Matsim_7-9/simulations/Matsim_7-9_Calibration__Original__Simulation_Dummy__14_16_1000/Matsim_7-9_Calibration__Original__None_None__None_None__None__0.00__0_0.00", "test", "3min")

In [ ]:
# affichage vitesse SPEED_MEAN et DISTANCE_DIFF en fonction du temps, 10 min

Utils.display_car_speed_distance_time("../../outputs/Matsim_7-9/simulations/Matsim_7-9_Calibration__Original__Simulation_Dummy__14_16_1000/Matsim_7-9_Calibration__Original__None_None__None_None__None__0.00__0_0.00", "test", "10min")

In [ ]:
# affichage vitesse SPEED_MEAN et DISTANCE_DIFF en fonction du temps, 20 min

Utils.display_car_speed_distance_time("../../outputs/Matsim_7-9/simulations/Matsim_7-9_Calibration__Original__Simulation_Dummy__14_16_1000/Matsim_7-9_Calibration__Original__None_None__None_None__None__0.00__0_0.00", "test", "20min")

## 6 - Bilan

Une simulation MnMS modélise le trajet de chaque vehicule de la manière suivante : à chaque pas de temps, un vehicule avance dans une section donnée d'une distance égale au produit du pas de temps (en secondes) et de la vitesse (en m/s, valeur déterminée par la fonction MFD en fonction de l'accumulation et du type de véhicule) sauf si cette dernière est plus importante que la distance qui lui reste à parcourir pour atteindre l'extrémité de la section dans laquelle il se trouve, le cas échéant le véhicule ne parcours que la distance pour atteindre l'extrémité de la section. La vitesse déterminée par le MFD, vitesse inscrite dans la colonne *SPEED* fichier de sortie *veh.csv*, n'est donc pas une juste approximation de la vitesse des véhicules en simulation. En revanche la variable *DISTANCE* de ce même fichier possède la distance réelle parcourue par les véhicules pendant la simulation. Cette modélisation des trajets est à l'origine des différences de valeurs entre vitesse calculée par la fonction MFD et vitesse calculée à partir de la distance parcourue : la vitesse calculée par MFD est surestimée car elle ne prend pas en compte le mécanisme "d'arrêt aux extrémités de chaque section" ou le principe de combinaison véhicule/section/pas de temps d'MnMS. Dans les parties suivantes sera donc privilégiée la méthode d'estimation de la vitesse par distance parcourue avec pas de temps petit mais adaptée à la méthode d'estimation des émissions via Copert (3 minutes). Un pas de temps trop élevé conduit à d'importantes erreurs, certains véhicules n'étant simuler que pendant des périodes très courtes.

Ces informations proviennent du code source d'MnMS : fichiers *mnms/vehicles/veh_type.py* fonction *update_distance* ligne 430 et *mnms/flox/MFD.py* fonction *move_veh* ligne 191-227;

In [ ]:
amsterdam_analysis_df.to_csv("../../outputs/AmsterdamCity_7-9/analysis/AmsterdamCity_7-9_Original__Original__Copert_Parameters_analysis/copert_parameters_analysis.csv", sep=';', index=False)